# 📊 انتخاب مدل در Climatology Engine

این نوت‌بوک معیارهای انتخاب بهترین مدل آماری را معرفی می‌کند.

**مواردی که یاد می‌گیرید:**
- معیار AIC (Akaike Information Criterion)
- معیار AICc (تصحیح شده برای حجم نمونه کوچک)
- معیار BIC (Bayesian Information Criterion)
- آزمون نسبت درست‌نمایی (Likelihood Ratio Test)
- تفسیر ΔAICc و وزن آکائیک (Akaike Weight)
- انتخاب بهترین مدل با چند معیار مختلف

---

## 📐 نظریه انتخاب مدل

### ۱. معیار اطلاعات آکائیک (AIC)

$$
AIC = 2k - 2\ln(\hat{L})
$$

که در آن:
- $k$: تعداد پارامترهای مدل
- $\hat{L}$: بیشینه درست‌نمایی (Maximum Likelihood)

### ۲. معیار AIC تصحیح شده (AICc)

$$
AIC_c = AIC + \frac{2k(k+1)}{n-k-1}
$$

که در آن $n$ حجم نمونه است. برای نمونه‌های کوچک ($n/k < 40$) توصیه می‌شود.

### ۳. معیار اطلاعات بیزی (BIC)

$$
BIC = k\ln(n) - 2\ln(\hat{L})
$$

### ۴. تفسیر ΔAICc

| ΔAICc | تفسیر |
|-------|-------|
| ≤ 2 | پشتیبانی قوی |
| 4-7 | پشتیبانی ضعیف |
| > 10 | عدم پشتیبانی |

### ۵. وزن آکائیک (Akaike Weight)

$$
w_i = \frac{\exp(-\Delta_i / 2)}{\sum_{j=1}^{m} \exp(-\Delta_j / 2)}
$$

که $\Delta_i = AICc_i - AICc_{min}$ است.

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from core.engine.plugin_loader import load_plugins

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11
print('✅ کتابخانه‌ها بارگذاری شدند.')

In [ ]:
# بارگذاری پلاگین‌های توزیع
plugins = load_plugins()
print(f'✅ تعداد توزیع‌های بارگذاری شده: {len(plugins)}')

for code, dist in plugins.items():
    print(f"   [{code}] {dist.name} (params: {dist.params})")

distributions = {dist.name: dist for dist in plugins.values()}

In [ ]:
# بارگذاری داده نمونه
sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
data = station_data.values

# انتخاب داده tmean برای یک سال
data_year = data[:365, 1]

print(f'📊 تعداد داده‌ها: {len(data_year)}')
print(f'   میانگین: {np.mean(data_year):.2f}°C')
print(f'   انحراف معیار: {np.std(data_year):.2f}°C')
print(f'   حجم نمونه (n): {len(data_year)}')

In [ ]:
# برازش همه توزیع‌ها
results_all = {}
print("\n🔄 در حال برازش توزیع‌ها...\n")

for name, dist in distributions.items():
    try:
        res = dist.fit(data_year)
        results_all[name] = res
        print(f"✅ {name}: AICc = {res.get('aicc', np.nan):.2f}, "
              f"BIC = {res.get('bic', np.nan):.2f}, "
              f"LogLik = {res.get('loglik', np.nan):.2f}")
    except Exception as e:
        print(f"❌ {name}: خطا - {str(e)}")
        results_all[name] = None

print("\n✅ برازش کامل شد.")

In [ ]:
# استخراج نتایج معتبر
valid_results = {k: v for k, v in results_all.items() 
                  if v is not None and 'aicc' in v and not np.isnan(v['aicc'])}

print(f"✅ تعداد مدل‌های معتبر: {len(valid_results)}")
print(f"   مدل‌ها: {list(valid_results.keys())}")

In [ ]:
# محاسبه معیارهای انتخاب مدل
def calculate_model_selection_metrics(results):
    """
    محاسبه معیارهای انتخاب مدل شامل:
    - AICc
    - ΔAICc
    - Akaike Weight
    - BIC
    - ΔBIC
    - BIC Weight
    """
    metrics = []
    
    # یافتن بهترین AICc و BIC
    min_aicc = min(r['aicc'] for r in results.values() if 'aicc' in r)
    min_bic = min(r['bic'] for r in results.values() if 'bic' in r and not np.isnan(r['bic']))
    
    for name, res in results.items():
        aicc = res.get('aicc', np.nan)
        bic = res.get('bic', np.nan)
        loglik = res.get('loglik', np.nan)
        n_params = res.get('n_params', np.nan)
        
        delta_aicc = aicc - min_aicc if not np.isnan(aicc) else np.nan
        delta_bic = bic - min_bic if not np.isnan(bic) else np.nan
        
        metrics.append({
            'مدل': name,
            'AICc': aicc,
            'ΔAICc': delta_aicc,
            'BIC': bic,
            'ΔBIC': delta_bic,
            'LogLik': loglik,
            'تعداد پارامتر': n_params
        })
    
    # محاسبه وزن آکائیک
    exp_vals = np.exp(-np.array([m['ΔAICc'] for m in metrics]) / 2)
    sum_exp = np.sum(exp_vals)
    for i, m in enumerate(metrics):
        m['وزن AICc'] = exp_vals[i] / sum_exp
    
    # محاسبه وزن BIC
    exp_vals_bic = np.exp(-np.array([m['ΔBIC'] for m in metrics]) / 2)
    sum_exp_bic = np.sum(exp_vals_bic)
    for i, m in enumerate(metrics):
        m['وزن BIC'] = exp_vals_bic[i] / sum_exp_bic
    
    return pd.DataFrame(metrics).sort_values('AICc').reset_index(drop=True)

selection_df = calculate_model_selection_metrics(valid_results)
selection_df.index = selection_df.index + 1

# نمایش جدول با فرمت زیبا
print("📊 جدول معیارهای انتخاب مدل:")
print("=" * 100)
selection_df.round(4)

In [ ]:
# نمایش بهترین مدل بر اساس AICc
best_aicc = selection_df.loc[selection_df['AICc'].idxmin()]
print("🏆 بهترین مدل بر اساس AICc:")
print("=" * 50)
print(f"   مدل: {best_aicc['مدل']}")
print(f"   AICc: {best_aicc['AICc']:.4f}")
print(f"   وزن AICc: {best_aicc['وزن AICc']:.4f} ({best_aicc['وزن AICc']*100:.1f}%)")
print(f"   تعداد پارامتر: {best_aicc['تعداد پارامتر']}")
print("=" * 50)

# نمایش بهترین مدل بر اساس BIC
best_bic = selection_df.loc[selection_df['BIC'].idxmin()]
print("\n🏆 بهترین مدل بر اساس BIC:")
print("=" * 50)
print(f"   مدل: {best_bic['مدل']}")
print(f"   BIC: {best_bic['BIC']:.4f}")
print(f"   وزن BIC: {best_bic['وزن BIC']:.4f} ({best_bic['وزن BIC']*100:.1f}%)")
print(f"   تعداد پارامتر: {best_bic['تعداد پارامتر']}")
print("=" * 50)

In [ ]:
# رسم نمودار وزن AICc مدل‌ها
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2ecc71' if i == 0 else '#e74c3c' if i == 1 else '#3498db' 
          for i in range(len(selection_df))]

bars = ax.bar(selection_df['مدل'], selection_df['وزن AICc'], 
              color=colors, alpha=0.7, edgecolor='black', linewidth=1)

ax.set_xlabel('مدل', fontsize=12)
ax.set_ylabel('وزن AICc', fontsize=12)
ax.set_title('وزن آکائیک (Akaike Weight) مدل‌های مختلف', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, weight in zip(bars, selection_df['وزن AICc']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{weight:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# رسم نمودار مقایسه ΔAICc
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2ecc71' if d == 0 else '#e74c3c' if d > 10 else '#f39c12' 
          for d in selection_df['ΔAICc']]

bars = ax.bar(selection_df['مدل'], selection_df['ΔAICc'], 
              color=colors, alpha=0.7, edgecolor='black', linewidth=1)

ax.axhline(y=2, color='green', linestyle='--', linewidth=2, alpha=0.7, label='ΔAICc = 2 (پشتیبانی قوی)')
ax.axhline(y=7, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='ΔAICc = 7 (پشتیبانی ضعیف)')
ax.axhline(y=10, color='red', linestyle='--', linewidth=2, alpha=0.7, label='ΔAICc = 10 (عدم پشتیبانی)')

ax.set_xlabel('مدل', fontsize=12)
ax.set_ylabel('ΔAICc', fontsize=12)
ax.set_title('مقایسه ΔAICc مدل‌ها', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

for bar, delta in zip(bars, selection_df['ΔAICc']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
            f'{delta:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# جدول تفسیر ΔAICc
def interpret_delta_aicc(delta):
    """تفسیر ΔAICc بر اساس راهنمای Burnham & Anderson"""
    if delta <= 2:
        return 'پشتیبانی قوی', 'green'
    elif delta <= 4:
        return 'پشتیبانی متوسط', 'lightgreen'
    elif delta <= 7:
        return 'پشتیبانی ضعیف', 'orange'
    elif delta <= 10:
        return 'پشتیبانی بسیار ضعیف', 'darkorange'
    else:
        return 'عدم پشتیبانی', 'red'

interpretation_df = selection_df[['مدل', 'ΔAICc', 'وزن AICc']].copy()
interpretation_df['تفسیر ΔAICc'], interpretation_df['رنگ'] = zip(*interpretation_df['ΔAICc'].apply(interpret_delta_aicc))

print("📋 تفسیر ΔAICc مدل‌ها:")
print("=" * 80)
interpretation_df[['مدل', 'ΔAICc', 'وزن AICc', 'تفسیر ΔAICc']]

In [ ]:
# رسم نمودار مقایسه AICc و BIC
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(selection_df['مدل']))
width = 0.35

bars1 = ax.bar(x - width/2, selection_df['AICc'], width, 
               label='AICc', color='#3498db', alpha=0.7, edgecolor='black', linewidth=1)
bars2 = ax.bar(x + width/2, selection_df['BIC'], width,
               label='BIC', color='#e74c3c', alpha=0.7, edgecolor='black', linewidth=1)

ax.set_xlabel('مدل', fontsize=12)
ax.set_ylabel('مقدار معیار', fontsize=12)
ax.set_title('مقایسه AICc و BIC مدل‌های مختلف', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(selection_df['مدل'])
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.5, f'{height:.1f}',
            ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.5, f'{height:.1f}',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# تابع برای انتخاب مدل با معیارهای مختلف
def select_best_model_by_criterion(df, criterion='AICc'):
    """
    انتخاب بهترین مدل بر اساس معیار مشخص
    criterion: 'AICc', 'BIC', 'LogLik', 'وزن AICc', 'وزن BIC'
    """
    if criterion in ['AICc', 'BIC']:
        best = df.loc[df[criterion].idxmin()]
    elif criterion in ['وزن AICc', 'وزن BIC']:
        best = df.loc[df[criterion].idxmax()]
    elif criterion == 'LogLik':
        best = df.loc[df[criterion].idxmax()]
    else:
        raise ValueError(f"معیار نامعتبر: {criterion}")
    return best

print("\n📊 انتخاب بهترین مدل با معیارهای مختلف:")
print("=" * 60)

criteria = ['AICc', 'BIC', 'LogLik', 'وزن AICc', 'وزن BIC']
for crit in criteria:
    try:
        best = select_best_model_by_criterion(selection_df, crit)
        print(f"\n✅ بر اساس {crit}:")
        print(f"   بهترین مدل: {best['مدل']}")
        print(f"   مقدار: {best[crit]:.4f}")
    except Exception as e:
        print(f"\n⚠️ خطا در محاسبه {crit}: {str(e)}")

print("\n" + "=" * 60)

## 📋 جمع‌بندی

در این نوت‌بوک یاد گرفتید:

✅ معیارهای انتخاب مدل: AIC، AICc، BIC
✅ محاسبه ΔAICc و تفسیر آن
✅ محاسبه وزن آکائیک (Akaike Weight)
✅ مقایسه مدل‌ها با معیارهای مختلف
✅ انتخاب بهترین مدل با چند معیار

---

**نکات کلیدی:**

1. **AICc** برای نمونه‌های کوچک مناسب‌تر از AIC است.
2. **وزن AICc** نشان‌دهنده احتمال اینکه مدل بهترین باشد، است.
3. **ΔAICc < 2** نشان‌دهنده پشتیبانی قوی از مدل است.
4. مدل‌های با **ΔAICc > 10** عملاً پشتیبانی نمی‌شوند.
5. **BIC** مدل‌های ساده‌تر را نسبت به AICc ترجیح می‌دهد.

---

**مراحل بعدی:**
- نوت‌بوک ۰۵: کنترل کیفیت (Quality Flag)
- نوت‌بوک ۰۶: عدم‌قطعیت Bootstrap
- نوت‌بوک ۰۷: پردازش موازی